## Lab 2 - Part B : Inverse Kinematics

### Overview & Variables

Part B  ให้เอาคำตอบ IK ที่แก้ **ด้วยมือใน Part A** มาตรวจด้วย Robotics Toolbox แล้ว cross-check ด้วย Forward Kinematics (FK)

| Variable | What is it? | From |
|--------|---------|-------------|
| `M` | ตำแหน่งเป้าหมายใน **base frame** ของหุ่น (= `Pobj`) หน่วยเมตร `[x, y, z]` | Part A ข้อ 1: แปลง `M` จาก camera frame มาเป็น base frame ด้วย inverse ของ `T` ที่ได้รับ |
| `q_goal` | มุม/ระยะข้อต่อที่แก้ IK **ด้วยมือ** (เลือกมา 1 branch) | Part A ข้อ 4 |
| `branches` | คำตอบ IK **ทุก branch** | Part A ข้อ 4 |

**หน่วยของ `q`** ขึ้นกับชนิดข้อต่อของหุ่นที่ออกแบบใน LAB1: revolute → เรเดียน `[rad]`, prismatic → เมตร `[m]`
(หุ่นใน template นี้เป็นแบบ **RRP** → `q1, q2` = rad, `q3` = m)

**เกณฑ์ผ่าน** error ในส่วนที่ 2 และ 4 ต้องไม่เกิน 0.03 m


### 0.1 Setup

In [ ]:
import numpy as np
from pathlib import Path
from roboticstoolbox import Robot
from spatialmath import SE3
import spatialgeometry as sg
import swift

HERE = Path.cwd().as_posix()
if hasattr(Robot, 'URDF_read'):             # older roboticstoolbox
    links, name = Robot.URDF_read('my_robot/robot.urdf', tld=HERE)[:2]
else:                                       # roboticstoolbox 1.5 and newer
    from roboticstoolbox.models.URDF import URDFRobot
    from xacrodoc.packages import update_package_cache
    update_package_cache({'my_robot': HERE + '/my_robot'})
    links, name = URDFRobot.URDF_read(HERE + '/my_robot/robot.urdf')[:2]
robot = Robot(links, name=name)
print('loaded', name, robot.n, 'joints')

env = swift.Swift()
env.launch(realtime=True, browser="notebook")
env.add(robot)
robot.q = [0.0, 0.0, 0.0]   # home configuration
env.step()

In [ ]:
# Allowable syntax (reference only - do not run as-is):
# robot.fkine(q)          # forward kinematics -> transform T ; tip position is T.t
# np.linalg.norm(a - b)   # distance between two points
# SE3(*M)                 # make a target pose from a point M = [x, y, z]
# robot.ikine_LM(SE3(*M), q0=..., mask=[1,1,1,0,0,0], joint_limits=True)   # numerical IK -> sol
# sol.q                   # the joint solution it found
# sol.success             # did it converge (True/False)

### 0.2 Cross-check your MDH model vs your URDF model

เช็คว่า URDF ตรงกับหุ่นที่คำนวณมือ (ตาราง MDH จาก LAB1) เพราะ Part A คำนวณ IK จากโมเดลที่คิดมือ

ต้องได้ `MDH vs URDF: OK` ถ้าไม่ผ่าน ให้หาว่าตาราง MDH หรือ URDF ผิด แล้วแก้ให้ตรงกันก่อนทำข้อ 1

In [ ]:
# Syntax (reference only):
# RevoluteMDH(a=..., alpha=..., d=..., offset=...)        # revolute joint; joint variable is the angle
# PrismaticMDH(theta=..., a=..., alpha=..., offset=...)   # prismatic joint; joint variable is the distance
# robot_mdh.tool = SE3(x, y, z)                           # end-effector offset from the last MDH frame
from roboticstoolbox import DHRobot, RevoluteMDH, PrismaticMDH

# edit here: your MDH table from LAB1 (use PrismaticMDH for a prismatic joint)
robot_mdh = DHRobot([
    RevoluteMDH(a=0.0, alpha=0.0, d=0.0),    # edit here : joint 1
    RevoluteMDH(a=0.0, alpha=0.0, d=0.0),    # edit here : joint 2
    RevoluteMDH(a=0.0, alpha=0.0, d=0.0),    # edit here : joint 3
], name='my_mdh')

robot_mdh.tool = SE3(0.0, 0.0, 0.0)          # edit here : end-effector offset (same as LAB1)
print(robot_mdh)

In [ ]:
# Cross-check (nothing to edit): at the same q both models must put the tip at the same point
lo, hi = robot.qlim
q_tests = [np.zeros(robot.n)] + [lo + (hi - lo) * np.roll([0.25, 0.6, 0.85], k)[:robot.n] for k in range(3)]

worst = 0.0
for q in q_tests:
    p_mdh, p_urdf = robot_mdh.fkine(q).t, robot.fkine(q).t
    diff = float(np.linalg.norm(p_mdh - p_urdf))
    worst = max(worst, diff)
    print('q        :', np.round(q, 4))
    print('MDH  tip :', np.round(p_mdh, 4))
    print('URDF tip :', np.round(p_urdf, 4))
    print('pos diff :', round(diff, 4), 'm')
    print('-----------------------------------')

if worst <= 0.005:
    print('MDH vs URDF: OK')
else:
    print('MDH vs URDF: NOT MATCHING - fix your MDH table or your URDF, then run 0.1 and 0.2 again')

### 1. target M (from Part A item 1) and q (from Part A item 4)

In [ ]:
M = np.array([0.0, 0.0, 0.0])       # edit here
q_goal = np.array([0.0, 0.0, 0.0])  # edit here

if np.allclose(M, 0) or np.allclose(q_goal, 0):
    print('fill in M and q_goal first, then run again')
else:
    marker = sg.Sphere(radius=0.03, color=(1.0, 0.1, 0.1, 0.6))
    marker.T = SE3(*M)
    env.add(marker)
    env.step()
    print('target shown at', np.round(M, 4))

In [ ]:
# move the robot to q_goal  (tip should reach the ball)
robot.q = q_goal
env.step()

### 2. cross-check with FK

In [ ]:
q = q_goal

tip = None    # edit here
dist = None   # edit here

if dist is None:
    print('fill in tip and dist above')
else:
    print('tip:', np.round(tip, 4), '| M:', np.round(M, 4), '| err:', round(float(dist), 4))

### 3. let the toolbox solve IK (ikine), compare with q_goal

In [ ]:
sol = None    # edit here  (see hint above)

if sol is None:
    print('fill in the ikine_LM call')
else:
    print('toolbox q:', np.round(sol.q, 4), '| ok', sol.success)
    print('hand q   :', np.round(q_goal, 4))

### 4. check every branch

In [ ]:
branches = [
    np.array([0.0, 0.0, 0.0]),    # edit here : branch 1
    # np.array([0.0, 0.0, 0.0]),  # branch 2 if any
]

for i, qb in enumerate(branches):
    err = np.linalg.norm(robot.fkine(qb).t - M)
    print('branch', i + 1, np.round(qb, 4), '-> err', round(float(err), 4))